# Reading & Writing Data — CSV, Excel, JSON, Parquet

## Definition
Pandas provides `read_*` and `to_*` functions for all major data formats.
Reading data efficiently is the **first step** of any ML pipeline.

### Supported Formats
| Format | Read | Write | Notes |
|--------|------|-------|-------|
| CSV | `pd.read_csv()` | `df.to_csv()` | Most common |
| Excel | `pd.read_excel()` | `df.to_excel()` | `.xlsx`, `.xls` |
| JSON | `pd.read_json()` | `df.to_json()` | Various orientations |
| Parquet | `pd.read_parquet()` | `df.to_parquet()` | Best for ML pipelines |
| SQL | `pd.read_sql()` | `df.to_sql()` | Requires SQLAlchemy |
| Feather | `pd.read_feather()` | `df.to_feather()` | Fast binary format |
| HTML | `pd.read_html()` | `df.to_html()` | Scraping tables |
| Clipboard | `pd.read_clipboard()` | `df.to_clipboard()` | Quick data entry |

In [ ]:
import pandas as pd
import numpy as np
import io, json, os, tempfile

# Create sample DataFrame for demos
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'age': [28, 34, 25, 31, 29],
    'salary': [75000.0, 85000.0, np.nan, 65000.0, 90000.0],
    'dept': ['Engineering', 'Marketing', 'Engineering', 'HR', 'Engineering'],
    'join_date': pd.to_datetime(['2020-01-15', '2019-03-22', '2021-07-01', '2018-11-30', '2022-02-14'])
})
print(df)
print(df.dtypes)

In [ ]:
# ─── CSV ───
# Write to CSV
csv_path = '/tmp/employees.csv'
df.to_csv(csv_path, index=False)  # index=False avoids writing row numbers
print('Written to:', csv_path)

# Read CSV
df_csv = pd.read_csv(csv_path)
print('\nRead back:')
print(df_csv)
print(df_csv.dtypes)

In [ ]:
# read_csv key parameters
# Simulating CSV content in-memory
csv_content = '''id,name,score,grade
1,Alice,85.5,A
2,Bob,,B
3,,78.0,C
4,Dave,92.0,A
'''

df_params = pd.read_csv(
    io.StringIO(csv_content),
    dtype={'id': int, 'score': float},   # force dtypes
    na_values=['', 'NA', 'N/A'],          # what counts as NaN
    usecols=['id', 'name', 'score'],      # load only specific columns
    skiprows=0,                           # skip N rows at top
)
print(df_params)

In [ ]:
# Read large CSV in chunks (for big files)
# This avoids loading the entire file into memory
chunk_path = '/tmp/employees.csv'
chunks = []
for chunk in pd.read_csv(chunk_path, chunksize=2):
    chunk['salary_processed'] = chunk['salary'].fillna(0)
    chunks.append(chunk)

df_chunked = pd.concat(chunks, ignore_index=True)
print('Processed chunks shape:', df_chunked.shape)

In [ ]:
# ─── JSON ───
json_path = '/tmp/employees.json'
df.to_json(json_path, orient='records', indent=2, date_format='iso')

df_json = pd.read_json(json_path, orient='records')
print('JSON read back:\n', df_json[['name', 'age', 'dept']].head())

In [ ]:
# ─── Parquet (recommended for ML pipelines!) ───
try:
    parquet_path = '/tmp/employees.parquet'
    df.to_parquet(parquet_path, index=False)
    df_parquet = pd.read_parquet(parquet_path)
    print('Parquet read back:')
    print(df_parquet.dtypes)
    print(df_parquet[['name', 'age', 'salary']].head())
except ImportError:
    print('Install pyarrow or fastparquet: pip install pyarrow')

In [ ]:
# ─── to_csv options ───
# Save with specific formatting
df.to_csv('/tmp/formatted.csv',
    index=False,
    float_format='%.2f',        # 2 decimal places for floats
    na_rep='MISSING',           # string for NaN values
    date_format='%Y-%m-%d',     # date format
    encoding='utf-8'
)

# Read and print
print(open('/tmp/formatted.csv').read())

In [ ]:
# EXCEPTION: File not found
try:
    pd.read_csv('/tmp/nonexistent_file.csv')
except FileNotFoundError as e:
    print('FileNotFoundError:', e)

# EXCEPTION: Parsing error
bad_csv = 'a,b,c\n1,2\n3,4,5,6'  # inconsistent columns
try:
    pd.read_csv(io.StringIO(bad_csv), on_bad_lines='error')
except Exception as e:
    print('ParserError:', type(e).__name__, str(e)[:80])

# Skip bad lines instead
df_safe = pd.read_csv(io.StringIO(bad_csv), on_bad_lines='skip')
print('After skipping bad lines:\n', df_safe)